In [45]:
import gymnasium as gym
import numpy as np

import torch
import torch.nn as nn


import random


In [2]:
# See available environments
print(gym.envs.registry.keys())

dict_keys(['CartPole-v0', 'CartPole-v1', 'MountainCar-v0', 'MountainCarContinuous-v0', 'Pendulum-v1', 'Acrobot-v1', 'phys2d/CartPole-v0', 'phys2d/CartPole-v1', 'phys2d/Pendulum-v0', 'LunarLander-v3', 'LunarLanderContinuous-v3', 'BipedalWalker-v3', 'BipedalWalkerHardcore-v3', 'CarRacing-v3', 'Blackjack-v1', 'FrozenLake-v1', 'FrozenLake8x8-v1', 'CliffWalking-v1', 'CliffWalkingSlippery-v1', 'Taxi-v4', 'tabular/Blackjack-v0', 'tabular/CliffWalking-v0', 'Reacher-v4', 'Reacher-v5', 'Pusher-v4', 'Pusher-v5', 'InvertedPendulum-v4', 'InvertedPendulum-v5', 'InvertedDoublePendulum-v4', 'InvertedDoublePendulum-v5', 'HalfCheetah-v4', 'HalfCheetah-v5', 'Hopper-v4', 'Hopper-v5', 'Swimmer-v4', 'Swimmer-v5', 'Walker2d-v4', 'Walker2d-v5', 'Ant-v4', 'Ant-v5', 'Humanoid-v4', 'Humanoid-v5', 'HumanoidStandup-v4', 'HumanoidStandup-v5', 'GymV21Environment-v0', 'GymV26Environment-v0', 'FetchSlide-v1', 'FetchSlide-v4', 'FetchPickAndPlace-v1', 'FetchPickAndPlace-v4', 'FetchReach-v1', 'FetchReach-v4', 'FetchPush-

In [54]:
env_name = "LunarLander-v3"

In [67]:
from typing import Any


class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, action_dim)        
        )

    def forward(self, x):
        return self.net(x)



class ValueNetwork(nn.Module):
    def __init__(self, state_dim) -> None:
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 100),
            nn.ReLU(),
            nn.Linear(100, 10),
            nn.ReLU(),
            nn.Linear(10, 1)
        )

    def forward(self, x):
        return self.net(x)

In [68]:
class ReplayBuffer:
    def __init__(self, size = 1000) -> None:
        self.N = size
        self.buffer = []

    def push(self, state, action, reward, new_state, done):
        if(len(self.buffer) > self.N):
            self.buffer.pop(0)
        self.buffer.append([state, action, reward, new_state, done])

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

In [100]:
def train(
    env,
    policy: PolicyNetwork,
    value: ValueNetwork,
    critic_criterion,
    policy_optimizer,
    value_optimizer,
    episodes=200,
    discount=0.9
):

    critic_losses = []
    actor_losses = []

    for i in range(episodes):

        observation, info = env.reset()

        done = False

        total_actor_loss = 0.0
        total_critic_loss = 0.0

        while not done:

            # -------------------------
            # 1. Select action
            # -------------------------

            state = torch.tensor(
                observation,
                dtype=torch.float32
            )

            action_logits = policy(state)

            action_dist = torch.distributions.Categorical(
                logits=action_logits
            )

            action = action_dist.sample()

            log_prob = action_dist.log_prob(action)


            # -------------------------
            # 2. Environment step
            # -------------------------

            new_observation, reward, terminated, truncated, info = env.step( action.item())

            done = terminated or truncated


            # -------------------------
            # 3. Calculate value
            # -------------------------

            value_state = value(state)

            new_state = torch.tensor(
                new_observation,
                dtype=torch.float32
            )

            with torch.no_grad():

                if done:
                    target = torch.tensor(
                        reward,
                        dtype=torch.float32
                    )

                else:
                    next_value = value(new_state)

                    target = reward + discount * next_value


            # -------------------------
            # 4. Critic loss
            # -------------------------

            critic_loss = critic_criterion( value_state, target )


            # -------------------------
            # 5. Advantage
            # -------------------------

            advantage = (
                target - value_state
            ).detach()


            # -------------------------
            # 6. Actor loss
            # -------------------------

            actor_loss = -(
                log_prob * advantage
            )


            # -------------------------
            # 7. Update networks
            # -------------------------

            policy_optimizer.zero_grad()
            value_optimizer.zero_grad()

            actor_loss.backward()
            critic_loss.backward()

            policy_optimizer.step()
            value_optimizer.step()


            # -------------------------
            # 8. Logging
            # -------------------------

            total_actor_loss += actor_loss.item()
            total_critic_loss += critic_loss.item()


            # Move to next state
            observation = new_observation


        critic_losses.append(total_critic_loss)
        actor_losses.append(total_actor_loss)

        print(
            f"episode: {i} | "
            f"critic loss: {total_critic_loss:.4f} | "
            f"actor loss: {total_actor_loss:.4f}"
        )

    return critic_losses, actor_losses

In [ ]:
def train_with_replay_buffer(env, policy: PolicyNetwork, value : ValueNetwork, critic_criterion, policy_optimizer,value_optimizer, episodes = 200, discount = 0.9):

    critic_losses = []
    actor_losses = []

    for i in range(episodes):
        observation, info = env.reset()

        done = False

        total_actor_loss = 0.0
        total_critic_loss = 0.0

        while not done:
            st = torch.tensor(observation, dtype=torch.float32)
            action_logits = policy()
            action_dist = torch.distributions.Categorical(logits=action_logits)
            at = action_dist.sample()

            new_observation, reward, terminated, truncated, info = env.step(at.item())

            
            vt = value(st)

            with torch.no_grad():
                stnew = torch.tensor(new_observation, dtype=torch.float32)
                rt = torch.tensor(reward, dtype=torch.float32)
                vtnew = value(stnew)

                if (done):
                    yt = rt
                else:
                    yt = rt + discount * vtnew 

            critic_loss = critic_criterion(vt, yt)


            action_logits = policy(st)
            action_dist = torch.distributions.Categorical(logits=action_logits)
            log_prob = action_dist.log_prob(at)

            actor_loss = -1 * (log_prob * (yt - vt).detach()).mean()

            policy_optimizer.zero_grad()
            value_optimizer.zero_grad()

            critic_loss.backward()
            actor_loss.backward()

            policy_optimizer.step()
            value_optimizer.step()

            total_critic_loss += critic_loss
            total_actor_loss += actor_loss

            observation = new_observation
            done = terminated or truncated

        critic_losses.append(total_critic_loss)
        actor_losses.append(total_actor_loss)

        print(f"iter: {i}  |  critic_loss: {total_critic_loss}  |  actor_loss: {total_actor_loss}")

    return critic_losses, actor_losses



In [101]:
def train_with_replay_buffer(env, policy: PolicyNetwork, value : ValueNetwork, replay_buffer : ReplayBuffer, critic_criterion, policy_optimizer,value_optimizer, episodes = 200, batch_size = 50, discount = 0.9):

    critic_losses = []
    actor_losses = []

    for i in range(episodes):
        observation, info = env.reset()

        done = False

        total_actor_loss = 0.0
        total_critic_loss = 0.0

        while not done:
            action_logits = policy(torch.tensor(observation, dtype=torch.float32))
            action_dist = torch.distributions.Categorical(logits=action_logits)
            action = action_dist.sample()

            new_observation, reward, terminated, truncated, info = env.step(action.numpy())

            replay_buffer.push(observation, action, reward, new_observation, done)

            if(len(replay_buffer) > batch_size ):

                batch = replay_buffer.sample(batch_size)

                st, at, rt, stnew, dt = zip(*batch)

                st = torch.tensor(st, dtype=torch.float32)
                at = torch.tensor(at, dtype=torch.float32)
                rt = torch.tensor(rt, dtype=torch.float32)
                stnew = torch.tensor(stnew, dtype=torch.float32)
                dt = torch.tensor(dt, dtype=torch.bool)

               
                vt = value(st)

                with torch.no_grad():
                    vtnew = value(stnew)
                    yt = rt + discount * vtnew * (1 - dt.float())

                critic_loss = critic_criterion(vt, yt)


                action_logits = policy(st)
                action_dist = torch.distributions.Categorical(logits=action_logits)
                log_prob = action_dist.log_prob(at)

                actor_loss = -1 * log_prob * (yt - vt).detach()

                policy_optimizer.zero_grad()
                value_optimizer.zero_grad()

                critic_loss.backward()
                actor_loss.backward()

                policy_optimizer.step()
                value_optimizer.step()

                total_critic_loss += critic_loss
                total_actor_loss += actor_loss

            observation = new_observation
            done = terminated or truncated

        critic_losses.append(total_critic_loss)
        actor_losses.append(total_actor_loss)

        print(f"iter: {i}  |  critic_loss: {total_critic_loss}  |  actor_loss: {total_actor_loss}")



In [102]:
state_dim = 8
action_dim = 4

policy = PolicyNetwork(state_dim, action_dim)
value = ValueNetwork(state_dim)
replay_buffer = ReplayBuffer(1000)

critic_criterion = nn.functional.mse_loss
policy_optimizer = torch.optim.Adam(policy.parameters())
value_optimizer = torch.optim.Adam(value.parameters())

env = gym.make(env_name)


In [104]:
train(env, policy, value, critic_criterion, policy_optimizer, value_optimizer)

C:\Users\Hasnain Ahmad\AppData\Local\Temp\ipykernel_6020\3063293452.py:84: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  critic_loss = critic_criterion( value_state, target )


episode: 0 | critic loss: 10316.7971 | actor loss: -172.1338
episode: 1 | critic loss: 10072.2422 | actor loss: -199.7192
episode: 2 | critic loss: 9272.9845 | actor loss: -289.5945
episode: 3 | critic loss: 10504.2277 | actor loss: -153.5294
episode: 4 | critic loss: 9409.3284 | actor loss: -250.6831
episode: 5 | critic loss: 8572.1277 | actor loss: 5.7210
episode: 6 | critic loss: 9294.3160 | actor loss: -96.6240
episode: 7 | critic loss: 8842.0981 | actor loss: -76.2360
episode: 8 | critic loss: 7285.0100 | actor loss: -197.6768
episode: 9 | critic loss: 8801.7762 | actor loss: 86.6241
episode: 10 | critic loss: 9275.7049 | actor loss: 102.6518
episode: 11 | critic loss: 9405.4151 | actor loss: 16.1481
episode: 12 | critic loss: 9590.3917 | actor loss: -31.6912
episode: 13 | critic loss: 9997.7921 | actor loss: -67.8048
episode: 14 | critic loss: 10138.9367 | actor loss: -170.2148
episode: 15 | critic loss: 9650.9685 | actor loss: -55.1422
episode: 16 | critic loss: 6613.2433 | acto

KeyboardInterrupt: 

In [93]:
observation, _, _, _, _ = env.step(env.action_space.sample())
observation

array([ 0.029982  ,  1.4480366 ,  0.76138735,  0.41047797, -0.02964899,
       -0.11919868,  0.        ,  0.        ], dtype=float32)

In [122]:
env = gym.make(env_name, render_mode = "human")

done = False

observation, info = env.reset()

while not done:
    action = policy(torch.tensor(observation)).argmax().numpy()
    observation, reward, t, tr, info = env.step(action)


    done = t or tr

env.close()

In [119]:
env = gym.make(env_name, render_mode = "human")

done = False

observation, info = env.reset()
action = policy(torch.tensor(observation)).argmax().numpy()
print(action)
env.close()

1


In [121]:
env.close()